# 主流RAG文档加载器

文档加载器主要工作是
1. 解析不同格式的原始文档，将PDF，Word，Markdown内容提取成可处理的纯文本
2. 解析过程中将文档来源，页码，作者等关键信息作为元数据
3. 将文本和元数据整理成统一的数据结构，方便后续切分，向量化和入库


| 工具名称 | 特点 | 适用场景 | 性能表现 |
| :--- | :--- | :--- | :--- |
| PyMuPDF4LLM | PDF→Markdown转换，OCR+表格识别 | 科研文献、技术手册 | 开源免费，GPU加速 |
| TextLoader | 基础文本文件加载 | 纯文本处理 | 轻量高效 |
| DirectoryLoader | 批量目录文件处理 | 混合格式文档库 | 支持多格式扩展 |
| Unstructured | 多格式文档解析 | PDF、Word、HTML等 | 统一接口，智能解析 |
| FireCrawlLoader | 网页内容抓取 | 在线文档、新闻 | 实时内容获取 |
| LlamaParse | 深度PDF结构解析 | 法律合同、学术论文 | 解析精度高，商业API |
| Docling | 模块化企业级解析 | 企业合同、报告 | IBM生态兼容 |
| Marker | PDF→Markdown，GPU加速 | 科研文献、书籍 | 专注PDF转换 |
| MinerU | 多模态集成解析 | 学术文献、财务报表 | 集成LayoutLMv3+YOLOv8 |

# Unstructured 文档处理库

Unstructured 是一个专业的文档处理库， 支持多种格式PDF、Word、Excel、HTML、Markdown 等多种文档格式，并通过统一的 API 接口避免为不同格式分别编写代码

所以unstructed就是用来处理文档的

LangChain的UnstructuredMarkdownLoader，它是 LangChain 对 Unstructured 库的封装。接下来展示如何直接使用 Unstructured 库，这样可以获得更大的灵活性和控制力。

**partition**函数使用自动文件类型检测，内部会根据文件类型路由到对应的专用函数（如PDF文件会调用partition_pdf

In [3]:
# 这个是unstructed 的 经典入口 
from unstructured.partition.auto import partition

pdf_path = "./data/C2/pdf/rag.pdf"

# 使用unstructed加载和分析pdf文档
elements = partition(
    filename=pdf_path,
    content_type="application/pdf"
)

print(f"解析完成: {len(elements)} 个元素, {sum(   len(str(e)) for e in elements   )} 字符")


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

short text: "Retrieval-augmented Generation". Defaulting to English.
short text: "RAG". Defaulting to English.
short text: "Printed using PDFCrowd". Defaulting to English.
short text: "HTML to PDF". Defaulting to English.
short text: "simayi315". Defaulting to English.
short text: "54hehezi". Defaulting to English.
short text: "Printed using PDFCrowd". Defaulting to English.
short text: "HTML to PDF". Defaulting to English.
short text: "Printed using PDFCrowd". Defaulting to English.
short text: "HTML to PDF". Defaulting to English.
short text: "Printed using PDFCrowd". Defaulting to English.
short text: "HTML to PDF". Defaulting to English.
short text: "5". Defaulting to English.
short text: "2". Defaulting to English.
short text: "4". Defaulting to English.
short text: "5". Defaulting to English.
short text: "6". Defaulting to English.
short text: "7". Defaulting to English.
short text: "8". Defaulting to English.
short text: "9". Defaulting to English.
short text: "10". Defaulting t

解析完成: 279 个元素, 7500 字符


In [5]:
# 下面统计元素类型
from collections import Counter
types = Counter(e.category for e in elements)
print(f"元素类型: {dict(types)}")


# 显示所有元素
print("\n所有元素:")
for i, element in enumerate(elements, 1):
    print(f"Element {i} ({element.category}):")
    print(element)
    print("=" * 60)

元素类型: {'Header': 22, 'Title': 195, 'UncategorizedText': 41, 'NarrativeText': 3, 'Footer': 15, 'ListItem': 3}

所有元素:
Element 1 (Header):
网页
Element 2 (Header):
新闻
Element 3 (Header):
贴吧
Element 4 (Header):
知道
Element 5 (Header):
网盘
Element 6 (Header):
图片
Element 7 (Header):
视频
Element 8 (Header):
地图
Element 9 (Header):
文库
Element 10 (Header):
资讯
Element 11 (Header):
采购
Element 12 (Header):
百科
Element 13 (Header):
百度首页 登录 注册
Element 14 (Title):
检索增强生成
Element 15 (Title):
进⼊词条
Element 16 (Title):
全站搜索
Element 17 (Title):
帮助
Element 18 (Title):
近期有不法分子冒充百度百科官方人员，以删除词条为由威胁并敲诈相关企业。在此严正声明：百度百科是免费编辑平台，绝不存在收费代编服务，请勿上当受骗！ 详情>>
Element 19 (Title):
首页
Element 20 (Title):
秒懂百科
Element 21 (Title):
特色百科
Element 22 (Title):
知识专题
Element 23 (Title):
加入百科
Element 24 (Title):
百科团队
Element 25 (Title):
权威合作
Element 26 (Title):
检索增强生成 播报 ⼤模型前沿技术之⼀ 展开2个同名词条
Element 27 (Title):
锁定
Element 28 (Title):
讨论
Element 29 (UncategorizedText):
1
Element 30 (Title):
上传视频
Element 31 (Title):
一分钟了解检索增强生成 一分钟了解检索增强生成
Elem